<a href="https://colab.research.google.com/github/tahiyat-rafi/Introduction-to-deep-learning-/blob/main/solving_a_1D_wave_equation_using_PINN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import torch
import torch.nn as nn

# pde: u_tt =(c**2) * u_xx
class pinn(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        return self.net(inputs)

#loss function
def loss(model, x, t, c=1.0):
    x.requires_grad_(True)
    t.requires_grad_(True)

    u = model(x, t)

    #diff
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]

    u_tt = torch.autograd.grad(u_t, t, grad_outputs=torch.ones_like(u_t), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

    err = u_tt - (c**2) * u_xx

    return torch.mean(err**2)

#training
model = pinn()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1000):
    optimizer.zero_grad()
    #sample points
    x_col = torch.rand(500, 1) * 2.0 - 1.0
    t_col = torch.rand(500, 1) * 1.0

    loss_val = loss(model, x_col, t_col)

    loss_val.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss_val.item():.6f}")

Epoch 0, Loss: 0.006244
Epoch 200, Loss: 0.000002
Epoch 400, Loss: 0.000000
Epoch 600, Loss: 0.000000
Epoch 800, Loss: 0.000000
